# Pro Kabaddi League player stats: load and inspect

Download the chosen Kaggle CSV into `data/raw/`, then update `CSV_FILENAME`, `DATASET_NAME`, `DATASET_URL`, `KEY_COLUMNS`, and `COLUMN_MEANINGS` in the setup cell below. The source, row count, key columns, and data dictionary are rendered from the actual CSV when the notebook runs. Verify column meanings against the Kaggle dataset description; they are intentionally not guessed.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

# Replace this with the CSV's exact filename in data/raw/.
CSV_FILENAME = "[filename].csv"
DATASET_NAME = "Pro Kabaddi League player statistics (enter exact Kaggle title)"
DATASET_URL = "[paste the Kaggle dataset URL]"

# Fill these in after inspecting the columns and the dataset documentation.
KEY_COLUMNS = []
COLUMN_MEANINGS = {
    # "column_name": "One-line meaning verified from the dataset documentation.",
}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RAW_DATA_DIR / CSV_FILENAME
if CSV_FILENAME == "[filename].csv":
    available_csvs = sorted((PROJECT_ROOT / "data" / "raw").glob("*.csv"))
    raise ValueError(
        "Set CSV_FILENAME to the downloaded file's exact name. CSV files currently in data/raw/: "
        + (", ".join(path.name for path in available_csvs) if available_csvs else "none")
    )
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}. Download it into data/raw/.")

df = pd.read_csv(CSV_PATH)
print(f"Loaded: {CSV_PATH}")

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nMissing values per column:")
display(df.isna().sum().rename("missing_values").to_frame())
print("\nFirst 10 rows:")
display(df.head(10))

In [ ]:
unknown_key_columns = sorted(set(KEY_COLUMNS) - set(df.columns))
if unknown_key_columns:
    raise ValueError(f"KEY_COLUMNS contains names not present in the CSV: {unknown_key_columns}")

def markdown_table(frame):
    headers = [str(column) for column in frame.columns]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    for row in frame.itertuples(index=False, name=None):
        values = [str(value).replace("|", "\\|").replace("\n", " ") for value in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

key_columns_text = ", ".join(KEY_COLUMNS) if KEY_COLUMNS else "Set KEY_COLUMNS after inspecting the CSV"
metadata = pd.DataFrame(
    [{
        "Dataset Name": DATASET_NAME,
        "Source": "Kaggle",
        "URL": DATASET_URL,
        "Number of rows": len(df),
        "Key columns": key_columns_text,
    }]
)
display(Markdown("## Dataset information\n\n" + markdown_table(metadata)))

In [ ]:
dictionary = pd.DataFrame({
    "Column": df.columns,
    "Meaning": [
        COLUMN_MEANINGS.get(column, "TODO: verify meaning from the Kaggle dataset documentation")
        for column in df.columns
    ],
    "Data type": [str(df[column].dtype) for column in df.columns],
})
display(Markdown("## Data dictionary\n\n" + markdown_table(dictionary)))

## Download the CSV from Kaggle

### Browser download

1. Open the dataset URL in a browser and sign in to Kaggle.
2. Click **Download** (or **Download All**). If it downloads a ZIP, extract it.
3. Copy the relevant `.csv` file into your project folder's `data/raw/` directory.
4. Set `CSV_FILENAME` above to the exact CSV filename, and set `DATASET_NAME` and `DATASET_URL` to the dataset's actual title and URL.
5. Inspect the columns, then fill in `KEY_COLUMNS` and every entry in `COLUMN_MEANINGS` using the dataset documentation. Run the cells from top to bottom.

### Kaggle CLI (Windows PowerShell)

In the VS Code terminal, from the project root, install the CLI if needed and authenticate using Kaggle's account settings/API token instructions:

```powershell
python -m pip install kaggle
kaggle datasets download -d <owner>/<dataset-slug> -p data/raw --unzip
```

Replace `<owner>/<dataset-slug>` with the identifier shown in the Kaggle dataset URL (for example, the portion after `kaggle.com/datasets/`). Do not share or commit your Kaggle API token. After download, use the CSV's exact filename for `CSV_FILENAME`.